# Nvidia: What Has to Be True?

A reverse-DCF walkthrough on NVDA using `valuationengine`.

## Thesis question

Nvidia trades at multiples that have rarely been seen in the megacap universe. Rather than guess whether it is "expensive," we use reverse DCF to make the embedded growth assumption explicit. The question becomes: **how long does Nvidia need to grow at hyperscaler-driven rates to justify the current price?**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from valuationengine.data.fetcher import fetch_company
from valuationengine.core.models import Assumptions
from valuationengine.core import dcf, reverse, sensitivity, scenario

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
nvda = fetch_company("NVDA")
print(f"Company: {nvda.name}")
print(f"Price: ${nvda.current_price:,.2f}")
print(f"Market cap: ${nvda.market_cap/1e9:,.1f}B")
print(f"Beta: {nvda.beta:.2f}")
print(f"Historical revenue CAGR: {nvda.historical_revenue_cagr*100:.1f}%")
print(f"Avg operating margin: {nvda.avg_operating_margin*100:.1f}%")

## 1. A higher-octane base case

Default assumptions are tuned for a mature business. For Nvidia, start with assumptions closer to its actual trajectory and let reverse DCF do the rest.

In [ ]:
a = Assumptions(
    revenue_growth=0.20,
    operating_margin=0.40,
    projection_years=7,
)
result = dcf.run(nvda, a)
print(result.summary())

## 2. Reverse DCF: how much growth does the price require?

In [ ]:
implied = reverse.solve(nvda, a, field="revenue_growth", bracket=(-0.05, 0.80))
print(implied["interpretation"])
print()
print(f"Implied revenue growth over {a.projection_years} years: {implied['implied_value']*100:.2f}%")

Hold this number against history. Nvidia's data-center revenue has grown faster than this in some recent windows; the question is durability, not whether it has ever happened. The reverse DCF reframes the debate from "is the price right" to "is this growth pace sustainable for this many years."

## 3. Sensitivity on margin and growth

In [ ]:
x_vals = [0.10, 0.15, 0.20, 0.25, 0.30]
y_vals = [0.30, 0.35, 0.40, 0.45, 0.50]
sens = sensitivity.run(
    nvda, a,
    x_field="revenue_growth", x_values=x_vals,
    y_field="operating_margin", y_values=y_vals,
)
sens

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(sens.values, aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(len(x_vals))); ax.set_xticklabels([f"{v*100:.0f}%" for v in x_vals])
ax.set_yticks(range(len(y_vals))); ax.set_yticklabels([f"{v*100:.0f}%" for v in y_vals])
ax.set_xlabel("Revenue growth"); ax.set_ylabel("Operating margin")
ax.set_title("NVDA intrinsic value per share")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 4. Scenarios

In [ ]:
scenarios = scenario.build_bull_base_bear(a, growth_delta=0.05, margin_delta=0.05)
results = scenario.run(nvda, scenarios)

rows = []
for name, r in results.items():
    rows.append({
        "scenario": name,
        "value_per_share": r.value_per_share,
        "current_price": nvda.current_price,
        "upside_pct": r.upside * 100,
    })
pd.DataFrame(rows)

## Takeaway

The market is paying for a specific trajectory: not just current growth, but its persistence. The reverse DCF puts a number on the bet. The sensitivity shows the payoff structure around that number. Whether to take the bet is, finally, a judgement about the durability of accelerated demand. The model is the scaffolding for the judgement, not the judgement itself.